# 08 — Density-ratio ablation: weight clipping and a stronger ridge penalty
Notebook 07 showed weight degeneracy: under the `sex` design the effective sample size (ESS) of the
calibration weights was about 13 out of 500, weighted methods returned unbounded regions for
12–20% of test points, and the EUc covariance became numerically degenerate.

This notebook compares four density-ratio variants **[my choice]**:

| Variant | What it is |
|---|---|
| `logistic` | the estimator used everywhere so far (logistic regression, ridge 1e-4) |
| `ridge10` | the same model with a strong ridge penalty (10) → flatter weights |
| `clip99` | `logistic`, weights capped at the 99% quantile of the training weights |
| `clip95` | `logistic`, weights capped at the 95% quantile |

Settings: the Gan & Liu tilting shift at k = 7 (record-level splits, as notebook 04) and the
`age` and `sex` patient designs of notebook 07 (patient-grouped calibration). Each split is fitted
ONCE and evaluated with every variant, so variant differences are paired.
Clipping trades bias (the weights no longer equal the ratio) for variance (higher ESS).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, os.path.abspath("../src"))
import cpshift as cp
import experiment as ex
import deep

print("device:", deep.DEVICE)

## Configuration (network and method settings as in notebooks 04 and 07)

In [ ]:
QUICK = True

CFG = dict(
    SEED=2026, ALPHA=0.10, N_TRAIN=1000, N_TEST=1000, N_REPS=20, B=1000, SCP_FIT_FRAC=0.5,
    DEEP_CAL_FRAC=0.5, DEEP_VAL_FRAC=0.2, HIDDEN=64, N_LAYERS=2, LR=1e-3, WEIGHT_DECAY=1e-4,
    BATCH=64, MAX_EPOCHS=500, PATIENCE=30, EPS=1e-3,
    OT_M=1024, OT_EPS=0.1, GRID_SIDE=150, GRCP_VOL_POINTS=100,
)
SETTINGS = [('tilt', 7), ('patient', 'age'), ('patient', 'sex')]
VARIANTS = ['logistic', 'ridge10', 'clip99', 'clip95']
CC = dict(WSC_DELTA=0.10, WSC_N_DIR=1000, WSC_GRID=50)
if QUICK:
    CFG.update(N_REPS=1, B=200, MAX_EPOCHS=40, GRCP_VOL_POINTS=20)
    CC.update(WSC_N_DIR=100)

OUT_DIR = "../results/ratio_ablation"
os.makedirs(OUT_DIR, exist_ok=True)
CFG

## Run

In [ ]:
df, X_all, Y_all = cp.load_parkinsons("../data/parkinsons_updrs.data")
res_df = ex.run_ratio_ablation(df, X_all, Y_all, SETTINGS, VARIANTS, CFG, CC, deep)
res_df.to_csv(f"{OUT_DIR}/results_all_reps.csv", index=False)

## Tables

In [ ]:
summ = (res_df.groupby(['setting', 'variant', 'method'])
        .agg(coverage=('coverage', 'mean'), coverage_sd=('coverage', 'std'),
             volume=('volume_median', 'median'), pct_infinite=('pct_infinite', 'mean'),
             WSC=('WSC', 'mean'), ess_cal=('ess_cal', 'mean'), n_cal=('n_cal', 'mean'))
        .round(2).reset_index())
summ.to_csv(f"{OUT_DIR}/summary.csv", index=False)
for setting in summ.setting.unique():
    s = summ[summ.setting == setting]
    tab = s.pivot(index='method', columns='variant', values=['coverage', 'volume', 'pct_infinite'])
    tab = tab.reindex(ex.ABLATION_METHODS)
    tab.to_csv(f"{OUT_DIR}/table_{setting.replace(':', '_')}.csv")
    print(f"\n=== {setting} | ESS by variant:", s.groupby('variant').ess_cal.first().to_dict(), "===")
    print(tab.to_string())

## Paired differences: each variant minus `logistic` (same splits)

In [ ]:
rows = []
for setting in res_df.setting.unique():
    for m in ['Deep-LocalCov + WCP (ours)', 'Deep-LocalCov + EUc (ours)', 'OTCP + WCP', 'EUc-LR']:
        d = res_df[(res_df.setting == setting) & (res_df.method == m)].pivot(
            index='rep', columns='variant', values=['coverage', 'pct_infinite'])
        for v in VARIANTS[1:]:
            for metric in ['coverage', 'pct_infinite']:
                x = (d[metric][v] - d[metric]['logistic']).to_numpy()
                p = stats.ttest_1samp(x, 0).pvalue if len(x) > 1 and np.std(x) > 0 else np.nan
                rows.append(dict(setting=setting, method=m, variant=v, metric=metric,
                                 mean_diff=x.mean(), n=len(x), p_t=p))
paired = pd.DataFrame(rows).round(4)
paired.to_csv(f"{OUT_DIR}/paired_vs_logistic.csv", index=False)
print(paired.to_string())

## Figure: coverage and % unbounded regions by variant

In [ ]:
show = ['Deep-LocalCov + WCP (ours)', 'Deep-LocalCov + EUc (ours)', 'OTCP + WCP', 'EUc-LR']
settings = list(summ.setting.unique())
fig, ax = plt.subplots(2, len(settings), figsize=(4.5 * len(settings), 7), squeeze=False)
xpos = np.arange(len(VARIANTS)); wbar = 0.8 / len(show)
for c, setting in enumerate(settings):
    s0 = summ[summ.setting == setting]
    for j, m in enumerate(show):
        s = s0[s0.method == m].set_index('variant').reindex(VARIANTS)
        ax[0, c].bar(xpos + j * wbar, s.coverage, wbar, label=m if c == 0 else None)
        ax[1, c].bar(xpos + j * wbar, s.pct_infinite, wbar)
    ax[0, c].axhline(90, color='k', lw=1, ls=':'); ax[0, c].set_ylim(60, 100)
    ax[0, c].set_title(setting)
    for r in range(2):
        ax[r, c].set_xticks(xpos + 0.4 - wbar / 2); ax[r, c].set_xticklabels(VARIANTS)
ax[0, 0].set_ylabel('coverage (%)'); ax[1, 0].set_ylabel('% unbounded regions')
fig.legend(loc='lower center', ncol=4, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_ratio_ablation.png", dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
print("Files saved in", os.path.abspath(OUT_DIR), ":", sorted(os.listdir(OUT_DIR)))
print("CONFIG:", CFG)